In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
SCALE_MODE = "log2FC"          # "log2FC" -> log2(HF/NF) | "index" -> (HF-NF)/(HF+NF)
CORRECTION_LABEL = "BH"        # Adjusted p-value method used in the summary table

DATA_PATH = Path("Combined_Master_Statistical_Summary.csv")
OUT_STEM = "PhenotypeEffect"

TREATMENT_ORDER = ["Ambient", "27°C", "32°C", "pH 7.8", "pH 7.6", "50m", "Dark"]
BLOCKS = [(0.5, "Control"), (1.5, "Temperature"),
          (3.5, "Acidification"), (5.5, "Light Limitation")]

# Canonical key -> publication label (mathtext)
TARGET_METRICS = {
    "settlement":       "Settlement Success (%)",
    "respiration":      r"Dark Respiration Rate ($\mathrm{nmol}\ \mathrm{O}_2\ \mathrm{mm}^{-3}\ \mathrm{min}^{-1}$)",
    "symbiont_density": r"Symbiont Density (cells $\mu\mathrm{m}^{-2}$)",
    "protein_density":  r"Protein Density ($\mu\mathrm{g}\ \mathrm{mm}^{-2}$)",
    "FvFm":             r"Photochemical Efficiency ($F_v/F_m$)",
    "Sigma":            r"Antenna Cross-Section ($\sigma_{\mathrm{PSII}}$, $\mathrm{\AA}^2$)",
    "Fm":             r"Max Fluorescence ($F_m$, A.U.)",
}

plt.rcParams.update({
    "font.family": "serif",
    "axes.edgecolor": "black",
    "axes.linewidth": 1.2,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "savefig.dpi": 600,
})

CMAP = LinearSegmentedColormap.from_list(
    "morph_advantage", ["#e57373", "#ffffff", "#81c784"], N=256
)

# ==============================================================================
# 2. LOADING & PREPARATION
# ==============================================================================
def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"ERROR: input table not found at '{path}'.")
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='latin1')

def build_group_means(summary_df, metric_csv_to_canonical):
    group_means_records = []

    for _, row in summary_df.iterrows():
        metric_label = row.get('Metric')
        condition = row.get('Condition')

        canonical_metric = metric_csv_to_canonical.get(metric_label)
        if canonical_metric is None:
            continue

        group_means_records.append(
            {
                'metric': canonical_metric,
                'condition': condition,
                'morph': 'HF',
                'mean': row.get('HF_Value', np.nan)
            }
        )
        group_means_records.append(
            {
                'metric': canonical_metric,
                'condition': condition,
                'morph': 'NF',
                'mean': row.get('NF_Value', np.nan)
            }
        )
    return pd.DataFrame(group_means_records)

def build_matrix(group_means):
    wide_mean = group_means.pivot_table(
        index="metric", columns=["condition", "morph"], values="mean", observed=True
    )

    vals = pd.DataFrame(index=list(TARGET_METRICS), columns=TREATMENT_ORDER, dtype=float)
    annot = pd.DataFrame(index=list(TARGET_METRICS), columns=TREATMENT_ORDER, dtype=object)

    for metric in TARGET_METRICS:
        for cond in TREATMENT_ORDER:
            try:
                m_hf = wide_mean.loc[metric, (cond, "HF")]
                m_nf = wide_mean.loc[metric, (cond, "NF")]
            except KeyError:
                vals.loc[metric, cond] = np.nan
                annot.loc[metric, cond] = "n.t."
                continue

            if pd.isna(m_hf) or pd.isna(m_nf) or m_hf <= 0 or m_nf <= 0:
                vals.loc[metric, cond] = np.nan
                annot.loc[metric, cond] = "n.t."
                continue

            if SCALE_MODE == "log2FC":
                val = np.log2(m_hf / m_nf)
            else:
                val = (m_hf - m_nf) / (m_hf + m_nf)

            vals.loc[metric, cond] = val
            annot.loc[metric, cond] = f"{'+' if val > 0 else ''}{val:.2f}"

    return vals, annot

def join_significance(annot, stats_source, scale_mode):
    if stats_source is None:
        return annot, False

    if isinstance(stats_source, Path):
        if not stats_source.exists():
            return annot, False
        stats_df = load_table(stats_source)
    elif isinstance(stats_source, pd.DataFrame):
        stats_df = stats_source
    else:
        raise TypeError("stats_source must be a Path or DataFrame.")

    required = {"metric", "condition", "p_adj"}
    if not required.issubset(stats_df.columns):
        return annot, False

    def tag(p):
        if pd.isna(p):
            return ""
        if p < 0.001:
            return "***"
        if p < 0.01:
            return "**"
        if p < 0.05:
            return "*"
        return ""

    lut = {(r.metric, r.condition): tag(r.p_adj) for r in stats_df.itertuples()}
    for metric in annot.index:
        for cond in annot.columns:
            t = lut.get((metric, cond), "")
            if t:
                annot.loc[metric, cond] = f"{annot.loc[metric, cond]}\n({t})"
    return annot, True

# ==============================================================================
# 3. RENDERING
# ==============================================================================
def render_heatmap(vals, annot, has_tags):
    plot_vals = vals.copy()
    plot_vals.index = [TARGET_METRICS[m] for m in plot_vals.index]
    plot_annot = annot.copy()
    plot_annot.index = plot_vals.index

    finite = plot_vals.values[np.isfinite(plot_vals.values.astype(float))]
    if finite.size > 0:
        abs_max = np.abs(finite).max()
    else:
        abs_max = 1.0 # Default if no finite values

    if SCALE_MODE == "log2FC":
        vlim = max(0.8, round(abs_max * 1.15, 1))
        cbar_label = r"$\log_2(\mathrm{HF}/\mathrm{NF})$"
        title = (
            r"Phenotype-associated differences across responses and conditions "
            r"($\log_2(\mathrm{HF}/\mathrm{NF})$, HF as numerator)"
        )
    else:
        vlim = 1.0
        cbar_label = r"$(\mathrm{HF}-\mathrm{NF})/(\mathrm{HF}+\mathrm{NF})$"
        title = "Phenotype effect: normalized difference index"

    # Cap the color range at +/- 1.0 as requested
    vlim = 1.0

    fig, ax = plt.subplots(figsize=(11.5, 7.0))

    sns.heatmap(
        plot_vals.astype(float),
        annot=plot_annot.values,
        fmt="",
        cmap=CMAP,
        vmin=-vlim,
        vmax=vlim,
        linewidths=0,
        linecolor="white",
        mask=~np.isfinite(plot_vals.astype(float)),
        annot_kws={"size": 10, "weight": "bold", "color": "black"},
        cbar_kws={"label": cbar_label, "shrink": 0.85, "aspect": 18, "pad": 0.03},
        ax=ax,
    )

    for i, _ in enumerate(plot_vals.index):
        for j, _ in enumerate(TREATMENT_ORDER):
            if not np.isfinite(plot_vals.iloc[i, j]):
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor="0.92",
                                           edgecolor="white", linewidth=0))

    # for x_pos, text in BLOCKS:
    #     ax.axvline(x_pos, color="black", linewidth=2.2)
    #     ax.text(x_pos, -0.14, text, transform=ax.get_xaxis_transform(),
    #             ha="center", fontsize=11.5, fontweight="bold", color="#333333")

    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.set_xticklabels(TREATMENT_ORDER, rotation=0, ha="center",
                       fontsize=11.5, fontweight="bold")
    ax.set_yticklabels(plot_vals.index, rotation=0, ha="right",
                       fontsize=11, fontweight="bold")
    ax.tick_params(axis="both", which="both", length=0)

    # ax.set_title(title, fontsize=13, fontweight="bold", pad=46)

    tag_line = (
        f"Significance tags: {CORRECTION_LABEL} adjusted p < 0.05 (*), < 0.01 (**), < 0.001 (***)"
        if has_tags else ""
    )

    # Move footer slightly higher
    plt.figtext(0.015, -0.055, tag_line, ha="left", va="top",
                fontsize=9, color="#666666", wrap=True)
    plt.figtext(0.985, -0.055, f"Footer note: 'n.t.' indicates no data for a given condition.", ha="right", va="top",
                fontsize=9, color="#666666", wrap=True)

    # Remove the existing figure title and place it as a figtext at the top center
    ax.set_title('')  # Remove the existing title
    plt.figtext(0.5, 0.96, title, ha="center", va="bottom",
                fontsize=13, fontweight="bold", color="black", wrap=True)

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(f"{OUT_STEM}.png", bbox_inches='tight')
    plt.savefig(f"{OUT_STEM}.pdf", bbox_inches='tight')
    plt.show()
